In [1]:
from transformers import pipeline
from huggingface_hub import login
from getpass import getpass

In [2]:
token = getpass("Please enter your Hugging Face API token: ")

if token:
    login(token=token)
    del token

Please enter your Hugging Face API token: ··········


In [3]:
pip install -q transformers datasets gradio accelerate

In [4]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# We will use TinyLlama-1.1B-Chat as it fits the criteria of an extremely small, efficient Llama-style model (1.1B parameters)
# and works exceptionally well on free Colab runtimes without OOM errors.
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Set up a conversational text generation pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# Simple system prompt and knowledge Base context injection for Indian Constitution and general legal advice
system_prompt = (
    "You are a helpful, professional, and accurate Indian Legal Advisor chatbot. "
    "Your task is to provide basic legal guidance based on the Constitution of India and general Indian legal frameworks. "
    "Always advise the user that this is for informational purposes and they should consult a professional lawyer for official matters."
)

def legal_advisor_bot(user_message, history):
    # Format the conversation using ChatML template used by TinyLlama
    messages = [{"role": "system", "content": system_prompt}]

    # Add conversation history
    for user_h, assistant_h in history:
        messages.append({"role": "user", "content": user_h})
        messages.append({"role": "assistant", "content": assistant_h})

    messages.append({"role": "user", "content": user_message})

    # Apply chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Generate response
    outputs = pipe(
        prompt,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )

    generated_text = outputs[0]["generated_text"]
    # Extract only the newly generated response assistant content
    response = generated_text[len(prompt):].strip()
    return response

In [ ]:
pip install -q peft trl bitsandbytes

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from transformers import TrainingArguments

# For demonstration, we load a small relevant legal QA dataset.
# You can swap this dataset with your specific Indian Constitution dataset path if you have a local JSON/CSV.
dataset = load_dataset("Ateeqq/legal-qa-dataset", split="train[:1000]") # Load subset for fast training

def format_prompts(batch):
    formatted_texts = []
    for q, a in zip(batch['question'], batch['answer']):
        # Formatting using the standard ChatML structure suitable for TinyLlama
        text = f"<|system|>\n{system_prompt}</s>\n<|user|>\n{q}</s>\n<|assistant|>\n{a}</s>"
        formatted_texts.append(text)
    return {"text": formatted_texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

In [ ]:
# Configure LoRA (Low-Rank Adaptation) to make training extremely lightweight and avoid Out-Of-Memory errors
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

training_args = TrainingArguments(
    output_dir="./legal_llama_results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=50, # Set low for demonstration; increase for full convergence
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)

# Train the model
trainer.train()

In [ ]:
# Update the existing pipeline to use the newly fine-tuned adapter
from peft import PeftModel

# Merge LoRA weights or load PEFT pipeline directly
fine_tuned_model = PeftModel.from_pretrained(model, "./legal_llama_results")
pipe = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer)

In [6]:
# Create and launch the Gradio UI
demo = gr.ChatInterface(
    fn=legal_advisor_bot,
    title="⚖️ Indian Legal Advisor Assistant",
    description="Ask questions regarding the Indian Constitution, fundamental rights, or general legal queries. Powered by TinyLlama.",
    examples=[
        "What are the Fundamental Rights guaranteed by the Indian Constitution?",
        "What is Article 21 of the Constitution of India?",
        "How can a citizen file a Public Interest Litigation (PIL)?"
    ]
)

# Launch with share=True so you can access it externally if needed
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f26e6fe51efa33133b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f26e6fe51efa33133b.gradio.live
